# LeetCode 3074 — Apple Redistribution into Boxes bằng Python

Link đề: https://leetcode.com/problems/apple-redistribution-into-boxes/ · Độ khó: **Easy** · Chủ đề: Array, Greedy, Sorting

Skeleton LeetCode:

```python
from typing import List

class Solution:
    def minimumBoxes(self, apple: List[int], capacity: List[int]) -> int:
        ...
```

Ghi chú: đề và các ví dụ được viết theo trí nhớ về đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code, nhưng nên đối chiếu lại với đề gốc nếu muốn khớp từng chữ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: có nhiều **gói táo** (`apple[i]` quả táo ở gói `i`) và nhiều **hộp** (`capacity[j]` là sức chứa của hộp `j`); cần **dồn tất cả táo vào ít hộp nhất**, biết táo của **cùng một gói được phép chia vào nhiều hộp khác nhau**.
2. Nắm công cụ: `sum`, `sorted(..., reverse=True)`, `itertools.accumulate` (tổng tích lũy), `heapq` (max-heap bằng số âm), mảng đếm theo giá trị, và `bisect` trên tổng tích lũy.
3. Đi từ cách **thử mọi tập hộp** (brute force) → **tham lam: sắp xếp hộp giảm dần, lấy hộp lớn trước** → **dùng heap** → **đếm theo giá trị sức chứa (counting)** → **tổng tích lũy + tìm nhị phân**.
4. Hiểu mấu chốt: vì táo chia tự do giữa các hộp, **chỉ cần tổng số táo**; và để dùng **ít hộp nhất** thì chọn **hộp có sức chứa lớn nhất trước** (lập luận tráo đổi).
5. Tránh các lỗi kinh điển: quên sắp xếp, sắp xếp tăng dần, dừng sớm/muộn một hộp (off-by-one với `>=` / `>`), tưởng mỗi gói phải nằm trọn trong một hộp.

## 2. Đề bài & ví dụ

Bạn được cho một mảng `apple` có độ dài `n` và một mảng `capacity` có độ dài `m`.

- Có `n` **gói táo**; gói thứ `i` chứa `apple[i]` quả táo.
- Có `m` **hộp**; hộp thứ `i` chứa được tối đa `capacity[i]` quả táo.

Hãy trả về **số hộp tối thiểu** cần chọn để **đặt lại toàn bộ táo từ `n` gói vào các hộp**.

**Lưu ý:** táo từ **cùng một gói** có thể được phân vào **các hộp khác nhau**. Mỗi quả táo vẫn chỉ nằm đúng một hộp.

**Ví dụ 1**

- Input: `apple = [1, 3, 2]`, `capacity = [4, 3, 1, 5, 2]`
- Output: `2`
- Giải thích: tổng số táo là `1 + 3 + 2 = 6`. Ta dùng hộp có sức chứa `4` và `5` (tổng `9 >= 6`); không thể dùng chỉ một hộp (hộp lớn nhất chỉ chứa `5 < 6`). Đáp án `2`.

**Ví dụ 2**

- Input: `apple = [5, 5, 5]`, `capacity = [2, 4, 2, 7]`
- Output: `4`
- Giải thích: tổng táo `15`. Ta phải dùng cả bốn hộp: `7 + 4 + 2 + 2 = 15`. Bỏ bất kỳ hộp nào cũng không đủ.

**Ràng buộc:** `1 <= n == apple.length <= 50`; `1 <= m == capacity.length <= 50`; `1 <= apple[i], capacity[i] <= 50`; đề đảm bảo có thể đặt hết táo vào các hộp.

In [1]:
from typing import List
from itertools import accumulate, combinations
import bisect
import heapq
import random

examples = [
    ([1, 3, 2], [4, 3, 1, 5, 2], 2),
    ([5, 5, 5], [2, 4, 2, 7], 4),
    ([2], [5], 1),
]
for apple, capacity, expected in examples:
    print("apple =", apple, "capacity =", capacity, "-> kỳ vọng", expected)

apple = [1, 3, 2] capacity = [4, 3, 1, 5, 2] -> kỳ vọng 2
apple = [5, 5, 5] capacity = [2, 4, 2, 7] -> kỳ vọng 4
apple = [2] capacity = [5] -> kỳ vọng 1


## 3. Kiến thức nền

### 3.1. `sum` và tổng số táo

`sum(apple)` cộng mọi phần tử. Với bài này, vì táo của mỗi gói có thể chia tự do vào các hộp, **chỉ có tổng số táo mới quan trọng** — việc chia gói nào ra sao không ảnh hưởng đến số hộp.

In [2]:
apple = [1, 3, 2]
total = sum(apple)
print("tổng số táo:", total)      # 6

tổng số táo: 6


### 3.2. Sắp xếp giảm dần: `sorted(x, reverse=True)`

`sorted(a)` trả về list **tăng dần**; thêm `reverse=True` được **giảm dần**. Ta muốn xét hộp **lớn nhất trước**. Cũng có `a.sort(reverse=True)` (sửa tại chỗ).

In [3]:
capacity = [4, 3, 1, 5, 2]
print(sorted(capacity))                    # [1, 2, 3, 4, 5]
print(sorted(capacity, reverse=True))      # [5, 4, 3, 2, 1]
print(capacity)                            # không đổi

[1, 2, 3, 4, 5]
[5, 4, 3, 2, 1]
[4, 3, 1, 5, 2]


### 3.3. Tổng tích lũy: `itertools.accumulate`

`accumulate([5, 4, 3, 2, 1])` cho dãy tổng dồn: `5, 9, 12, 14, 15` (phần tử thứ `i` là tổng của `i + 1` số đầu). Nhờ đó ta biết nhanh "dùng `k` hộp lớn nhất chứa được bao nhiêu táo".

In [4]:
caps_desc = [5, 4, 3, 2, 1]
prefix = list(accumulate(caps_desc))
print(prefix)          # [5, 9, 12, 14, 15]
print(prefix[1])       # 9: hai hộp lớn nhất chứa được 9 quả

[5, 9, 12, 14, 15]
9


### 3.4. `for ... break` và `enumerate(start=1)`

Khi cần "dừng ngay khi đủ", ta dùng `break` để thoát vòng lặp. `enumerate(a, start=1)` đánh số từ `1`, rất tiện để biết ta đã dùng **bao nhiêu** hộp (số lượng, không phải chỉ số từ 0).

In [5]:
for count, c in enumerate([5, 4, 3], start=1):
    print("đã dùng", count, "hộp, vừa thêm hộp", c)
    if count == 2:
        break

đã dùng 1 hộp, vừa thêm hộp 5
đã dùng 2 hộp, vừa thêm hộp 4


### 3.5. Max-heap bằng số âm (`heapq`)

`heapq` của Python là **min-heap**. Muốn lấy phần tử **lớn nhất** ta đẩy vào **số âm**: `heappush(h, -x)`, `-heappop(h)`. Ta sẽ dùng cho Cách 3.

In [6]:
h = [-c for c in [4, 3, 1, 5, 2]]
heapq.heapify(h)
print(-heapq.heappop(h), -heapq.heappop(h))     # 5 4 : lớn nhất rồi lớn nhì

5 4


### 3.6. Mảng đếm theo giá trị (counting)

Vì `capacity[i] <= 50`, ta có thể đếm "có bao nhiêu hộp mỗi sức chứa" trong mảng `count[v]` (với `v` từ `0` đến `50`), rồi duyệt **từ `50` xuống `1`** để lấy hộp lớn trước, không cần sắp xếp.

In [7]:
capacity = [4, 3, 1, 5, 2, 5]
count = [0] * 51
for c in capacity:
    count[c] += 1
print({v: n for v, n in enumerate(count) if n})      # {1: 1, 2: 1, 3: 1, 4: 1, 5: 2}

{1: 1, 2: 1, 3: 1, 4: 1, 5: 2}


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Lấy `apple = [1, 3, 2]`, `capacity = [4, 3, 1, 5, 2]`. Tổng táo là `6`.

Ta thử **chọn từng số hộp**:

| số hộp dùng | chọn hộp nào (để chứa nhiều nhất) | tổng sức chứa | đủ chứa 6? |
|---|---|---|---|
| 1 | `5` | 5 | không |
| 2 | `5, 4` | 9 | **có** |

Với mỗi số hộp `k`, cách tốt nhất để chứa được nhiều táo là chọn **`k` hộp lớn nhất**. Do đó câu trả lời là **`k` nhỏ nhất** sao cho tổng `k` hộp lớn nhất `>= 6`. Ở đây `k = 2`.

In [8]:
apple = [1, 3, 2]
capacity = [4, 3, 1, 5, 2]
total = sum(apple)
caps = sorted(capacity, reverse=True)
running = 0
for k, c in enumerate(caps, start=1):
    running += c
    print(f"dùng {k} hộp lớn nhất {caps[:k]} -> tổng {running}, cần {total}, đủ? {running >= total}")
    if running >= total:
        print("đáp án =", k)
        break

dùng 1 hộp lớn nhất [5] -> tổng 5, cần 6, đủ? False
dùng 2 hộp lớn nhất [5, 4] -> tổng 9, cần 6, đủ? True
đáp án = 2


## 5. Cách 1 — Brute Force / baseline: thử mọi tập hộp

**Ý tưởng.** Với `k = 1, 2, ..., m`: thử **mọi cách chọn `k` hộp** (`itertools.combinations`), nếu có cách nào có tổng sức chứa `>= tổng táo` thì `k` là đáp án (trả về ngay vì ta thử `k` tăng dần).

Cách này đúng theo định nghĩa "số hộp tối thiểu" nhưng dùng **sức mạnh thô**: số cách chọn tăng theo `2^m`.

In [9]:
def minimum_boxes_bruteforce(apple: List[int], capacity: List[int]) -> int:
    total = sum(apple)
    m = len(capacity)
    for k in range(1, m + 1):                         # thử số hộp tăng dần
        for chosen in combinations(capacity, k):      # mọi cách chọn k hộp
            if sum(chosen) >= total:
                return k
    return -1                                         # không xảy ra theo đề


for apple, capacity, expected in examples:
    print(minimum_boxes_bruteforce(apple, capacity), "(kỳ vọng", expected, ")")

2 (kỳ vọng 2 )
4 (kỳ vọng 4 )
1 (kỳ vọng 1 )


**Độ phức tạp (Cách 1).** Tổng số tập hộp là `2^m - 1`; mỗi tập tốn `O(m)` để cộng ⇒ **Time `O(2^m · m)`** — với `m = 50` là `~10^15`, **không chạy nổi**. **Space `O(m)`**.

**Nút thắt:** ta thử cả những tập hộp "kém" (chọn hộp nhỏ trong khi còn hộp lớn chưa dùng). Nếu chứng minh được **chỉ cần xét tập gồm `k` hộp lớn nhất**, thì chỉ có `m` tập cần thử.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước 1: chỉ cần tổng táo.** Vì táo mỗi gói chia được vào nhiều hộp, ta có thể coi mọi táo là **một đống `total = sum(apple)` quả**. Yêu cầu: chọn tập hộp có **tổng sức chứa `>= total`**, với số hộp ít nhất.

**Bước 2: tham lam, chọn hộp lớn trước (lập luận tráo đổi).** Giả sử có đáp án tối ưu dùng `k` hộp mà trong đó có hộp `A` nhỏ hơn hộp `B` **chưa được dùng**. Thay `A` bằng `B`: tổng sức chứa không giảm (`B >= A`), số hộp vẫn `k`. Lặp lại, ta được đáp án tối ưu gồm **`k` hộp lớn nhất**.

**Kết luận:** sắp xếp `capacity` **giảm dần**, cộng dồn từ đầu cho tới khi tổng `>= total`; số hộp đã cộng là đáp án.

In [10]:
# Kiểm chứng bước 2 bằng brute force nhỏ: tập tốt nhất cỡ k luôn có tổng = tổng k hộp lớn nhất
rng = random.Random(11)
for _ in range(500):
    caps = [rng.randint(1, 50) for _ in range(rng.randint(1, 8))]
    for k in range(1, len(caps) + 1):
        best = max(sum(c) for c in combinations(caps, k))
        assert best == sum(sorted(caps, reverse=True)[:k])
print("với mọi k, tổng lớn nhất của k hộp = tổng k hộp lớn nhất (đã kiểm chứng)")

với mọi k, tổng lớn nhất của k hộp = tổng k hộp lớn nhất (đã kiểm chứng)


## 7. Các cách tiếp theo

### Cách 2 — Tham lam: sắp xếp giảm dần rồi cộng dồn

**Giải thích từng đoạn.**

1. `total = sum(apple)`: tổng số táo cần chứa.
2. `capacity.sort(reverse=True)` (trên bản sao): hộp lớn nhất đứng đầu.
3. Duyệt từng hộp, cộng sức chứa vào `running`; mỗi lần cộng, đếm thêm một hộp.
4. Ngay khi `running >= total`, trả về số hộp đã dùng.

In [11]:
def minimum_boxes_greedy(apple: List[int], capacity: List[int]) -> int:
    total = sum(apple)
    caps = sorted(capacity, reverse=True)       # hộp lớn trước
    running = 0
    for count, c in enumerate(caps, start=1):
        running += c                            # dùng thêm hộp này
        if running >= total:                    # đủ chỗ cho mọi quả táo
            return count
    return -1


for apple, capacity, expected in examples:
    print(minimum_boxes_greedy(apple, capacity), "(kỳ vọng", expected, ")")

2 (kỳ vọng 2 )
4 (kỳ vọng 4 )
1 (kỳ vọng 1 )


In [12]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
class SolutionGreedy:
    def minimumBoxes(self, apple: List[int], capacity: List[int]) -> int:
        total = sum(apple)
        running = 0
        for count, c in enumerate(sorted(capacity, reverse=True), start=1):
            running += c
            if running >= total:
                return count
        return -1


print(SolutionGreedy().minimumBoxes([5, 5, 5], [2, 4, 2, 7]))     # 4

4


**Độ phức tạp Cách 2:** sắp xếp `O(m log m)`, duyệt `O(m)`, tính tổng `O(n)` ⇒ **Time `O(n + m log m)`**, **Space `O(m)`** (bản sắp xếp).

### Cách 3 — Max-heap

**Giải thích.** Thay vì sắp xếp cả mảng, ta đẩy mọi sức chứa vào max-heap rồi **lấy lần lượt lớn nhất** cho tới khi đủ. Hữu ích khi chỉ cần **vài** hộp đầu (nhưng worst case vẫn `O(m log m)`).

In [13]:
def minimum_boxes_heap(apple: List[int], capacity: List[int]) -> int:
    total = sum(apple)
    heap = [-c for c in capacity]               # số âm để dùng min-heap như max-heap
    heapq.heapify(heap)
    running = 0
    count = 0
    while running < total:
        running += -heapq.heappop(heap)         # lấy hộp lớn nhất còn lại
        count += 1
    return count


for apple, capacity, expected in examples:
    print(minimum_boxes_heap(apple, capacity), "(kỳ vọng", expected, ")")

2 (kỳ vọng 2 )
4 (kỳ vọng 4 )
1 (kỳ vọng 1 )


**Độ phức tạp Cách 3:** `heapify` `O(m)`, mỗi `heappop` `O(log m)`, tối đa `m` lần ⇒ **Time `O(n + m log m)`** (tệ nhất), **Space `O(m)`**.

### Cách 4 — Đếm theo giá trị (counting), không sắp xếp

**Giải thích.** Vì `capacity[i] <= 50`, tạo mảng `count[0..50]`. Duyệt `v` từ `50` xuống `1`; mỗi giá trị `v` có `count[v]` hộp; ta lấy hộp loại `v` **từng chiếc một** cho tới khi đủ. Cách này chạy `O(n + m + C)` với `C = 50` là giá trị tối đa — **không phụ thuộc `log`**.

In [14]:
def minimum_boxes_counting(apple: List[int], capacity: List[int]) -> int:
    total = sum(apple)
    MAXV = 50
    count = [0] * (MAXV + 1)
    for c in capacity:
        count[c] += 1                           # đếm số hộp theo sức chứa
    running = 0
    boxes = 0
    for v in range(MAXV, 0, -1):                # từ lớn xuống nhỏ
        for _ in range(count[v]):               # lấy từng hộp loại v
            running += v
            boxes += 1
            if running >= total:
                return boxes
    return -1


for apple, capacity, expected in examples:
    print(minimum_boxes_counting(apple, capacity), "(kỳ vọng", expected, ")")

2 (kỳ vọng 2 )
4 (kỳ vọng 4 )
1 (kỳ vọng 1 )


**Độ phức tạp Cách 4:** **Time `O(n + m + C)`** với `C = 50`, **Space `O(C)`**. Rất hợp khi giá trị bị chặn nhỏ.

### Cách 5 — Tổng tích lũy + tìm nhị phân

**Giải thích.** Sắp xếp giảm dần rồi tính `prefix = accumulate(caps)`; `prefix` **tăng dần** (mọi sức chứa dương). Số hộp tối thiểu là **chỉ số `i` đầu tiên có `prefix[i] >= total`, cộng 1**. `bisect_left(prefix, total)` cho chính chỉ số đó bằng tìm nhị phân.

In [15]:
def minimum_boxes_bisect(apple: List[int], capacity: List[int]) -> int:
    total = sum(apple)
    caps = sorted(capacity, reverse=True)
    prefix = list(accumulate(caps))             # prefix[i] = tổng i+1 hộp lớn nhất
    i = bisect.bisect_left(prefix, total)       # chỉ số đầu tiên có prefix[i] >= total
    return i + 1


for apple, capacity, expected in examples:
    print(minimum_boxes_bisect(apple, capacity), "(kỳ vọng", expected, ")")

2 (kỳ vọng 2 )
4 (kỳ vọng 4 )
1 (kỳ vọng 1 )


**Độ phức tạp Cách 5:** **Time `O(n + m log m)`** (sắp xếp), **Space `O(m)`**. Phần tìm nhị phân chỉ `O(log m)`; cách này đáng học vì dạng "tổng tích lũy + bisect" lặp lại ở nhiều bài.

## 8. TRACE từng bước

Ta trace Cách 2 (tham lam) trên ví dụ 2 và Cách 4 (đếm) trên ví dụ 1.

In [16]:
def trace_greedy(apple: List[int], capacity: List[int]) -> int:
    total = sum(apple)
    caps = sorted(capacity, reverse=True)
    print(f"apple={apple} -> tổng táo = {total}")
    print(f"capacity sắp xếp giảm dần = {caps}")
    running = 0
    for count, c in enumerate(caps, start=1):
        running += c
        done = running >= total
        print(f"hộp #{count}: sức chứa {c} -> tổng sức chứa = {running} {'>=' if done else '<'} {total}")
        if done:
            print("đủ chỗ -> đáp án =", count)
            return count
    return -1


assert trace_greedy([5, 5, 5], [2, 4, 2, 7]) == 4

apple=[5, 5, 5] -> tổng táo = 15
capacity sắp xếp giảm dần = [7, 4, 2, 2]
hộp #1: sức chứa 7 -> tổng sức chứa = 7 < 15
hộp #2: sức chứa 4 -> tổng sức chứa = 11 < 15
hộp #3: sức chứa 2 -> tổng sức chứa = 13 < 15
hộp #4: sức chứa 2 -> tổng sức chứa = 15 >= 15
đủ chỗ -> đáp án = 4


In [17]:
def trace_counting(apple: List[int], capacity: List[int]) -> int:
    total = sum(apple)
    count = [0] * 51
    for c in capacity:
        count[c] += 1
    print("tổng táo =", total, "| số hộp theo sức chứa:", {v: n for v, n in enumerate(count) if n})
    running = boxes = 0
    for v in range(50, 0, -1):
        for _ in range(count[v]):
            running += v
            boxes += 1
            print(f"lấy hộp sức chứa {v}: boxes={boxes}, running={running}")
            if running >= total:
                print("đủ -> đáp án =", boxes)
                return boxes
    return -1


assert trace_counting([1, 3, 2], [4, 3, 1, 5, 2]) == 2

tổng táo = 6 | số hộp theo sức chứa: {1: 1, 2: 1, 3: 1, 4: 1, 5: 1}
lấy hộp sức chứa 5: boxes=1, running=5
lấy hộp sức chứa 4: boxes=2, running=9
đủ -> đáp án = 2


Bảng trace Cách 2 với ví dụ 2 (`total = 15`, hộp giảm dần `[7, 4, 2, 2]`):

| hộp thứ | sức chứa | tổng sức chứa (running) | `running >= 15`? |
|---|---|---|---|
| 1 | 7 | 7 | không |
| 2 | 4 | 11 | không |
| 3 | 2 | 13 | không |
| 4 | 2 | 15 | **có** → đáp án 4 |

## 9. Code nộp LeetCode

Bản nộp dùng Cách 2 (tham lam, sắp xếp giảm dần): ngắn, rõ, `O(n + m log m)`.

In [18]:
from typing import List

class Solution:
    def minimumBoxes(self, apple: List[int], capacity: List[int]) -> int:
        total = sum(apple)                                  # tổng số táo
        running = 0
        for count, c in enumerate(sorted(capacity, reverse=True), start=1):
            running += c                                    # dùng thêm hộp lớn nhất còn lại
            if running >= total:                            # đủ chứa mọi quả táo
                return count
        return -1                                           # theo đề không bao giờ xảy ra


sol = Solution()
for apple, capacity, expected in examples:
    print(apple, capacity, "->", sol.minimumBoxes(apple, capacity))

[1, 3, 2] [4, 3, 1, 5, 2] -> 2
[5, 5, 5] [2, 4, 2, 7] -> 4
[2] [5] -> 1


## 10. Vì sao thuật toán đúng

- **Đưa về tổng:** táo từ một gói có thể chia vào các hộp khác nhau, nên một cách đặt hợp lệ tồn tại ⇔ tổng sức chứa các hộp được chọn `>= sum(apple)`. (Đặt táo lần lượt vào hộp còn chỗ cho tới hết.)
- **Chọn `k` hộp lớn nhất là tối ưu cho mỗi `k`:** nếu một tập `k` hộp có hộp `A` mà còn hộp `B > A` chưa chọn, đổi `A` thành `B` không làm giảm tổng. Lặp lại tới khi tập là `k` hộp lớn nhất.
- **Tìm `k` nhỏ nhất:** tổng `k` hộp lớn nhất tăng theo `k`, nên `k` đầu tiên thỏa `>= total` là số hộp tối thiểu. Nếu `k - 1` hộp lớn nhất chưa đủ thì mọi tập `k - 1` hộp đều chưa đủ.
- **Đảm bảo có đáp án:** đề cam kết tổng mọi hộp `>= total`, nên vòng lặp luôn dừng bằng `return count` trước khi hết hộp.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(apple)`, `m = len(capacity)`, `C = 50` là sức chứa tối đa.

| Cách | Time | Space |
|---|---|---|
| 1. Thử mọi tập hộp | `O(2^m · m)` | `O(m)` |
| 2. Sắp xếp giảm dần + cộng dồn | `O(n + m log m)` | `O(m)` |
| 3. Max-heap | `O(n + m log m)` | `O(m)` |
| 4. Đếm theo giá trị | `O(n + m + C)` | `O(C)` |
| 5. Tổng tích lũy + `bisect` | `O(n + m log m)` | `O(m)` |

Với `n, m <= 50`, mọi cách trừ Cách 1 đều dưới mili-giây.

## 12. Bộ test

Ta gom các cách vào `impls`; Cách 1 (brute force) chỉ dùng với `m` nhỏ.

In [19]:
sol = Solution()
impls = {
    "greedy": minimum_boxes_greedy,
    "heap": minimum_boxes_heap,
    "counting": minimum_boxes_counting,
    "bisect": minimum_boxes_bisect,
    "leetcode": sol.minimumBoxes,
}


def check_all(apple, capacity, expected, label):
    for key, fn in impls.items():
        a, c = list(apple), list(capacity)
        actual = fn(a, c)
        assert actual == expected, (label, key, apple, capacity, actual, expected)
        assert a == apple and c == capacity, ("hàm làm thay đổi input", key)
    if len(capacity) <= 16:
        assert minimum_boxes_bruteforce(list(apple), list(capacity)) == expected, (label, "brute")
    print("PASS", label)


for idx, (apple, capacity, expected) in enumerate(examples, 1):
    check_all(apple, capacity, expected, f"ví dụ {idx}")

PASS ví dụ 1
PASS ví dụ 2
PASS ví dụ 3


### 12.1. Các trường hợp biên

In [20]:
check_all([1], [1], 1, "một gói một hộp, vừa khít")
check_all([1], [50, 1], 1, "một hộp lớn là đủ")
check_all([3, 3], [3, 3], 2, "tổng 6 = 3 + 3, cần cả hai hộp")
check_all([3, 3], [6, 1], 1, "hộp 6 đủ chứa tổng 6 (bằng, không cần lớn hơn)")
check_all([50] * 50, [50] * 50, 50, "tổng 2500, cần đủ 50 hộp 50")
check_all([1] * 50, [50], 1, "50 gói mỗi gói 1 quả, một hộp 50")
check_all([10], [1, 2, 3, 4, 5], 3, "5 + 4 = 9 < 10 nên cần 3 hộp: 5 + 4 + 3 = 12")
check_all([1, 2, 3, 4], [5, 5, 5], 2, "tổng 10: hai hộp 5 + 5")
check_all([6], [1, 2, 3], 3, "tổng 6 = 3 + 2 + 1: cần cả ba hộp")
check_all([2, 2, 2], [6, 6], 1, "tổng 6: một hộp 6")
check_all([5, 5, 5], [2, 4, 2, 7], 4, "ví dụ 2: phải dùng hết bốn hộp")

PASS một gói một hộp, vừa khít
PASS một hộp lớn là đủ
PASS tổng 6 = 3 + 3, cần cả hai hộp
PASS hộp 6 đủ chứa tổng 6 (bằng, không cần lớn hơn)
PASS tổng 2500, cần đủ 50 hộp 50
PASS 50 gói mỗi gói 1 quả, một hộp 50
PASS 5 + 4 = 9 < 10 nên cần 3 hộp: 5 + 4 + 3 = 12
PASS tổng 10: hai hộp 5 + 5
PASS tổng 6 = 3 + 2 + 1: cần cả ba hộp
PASS tổng 6: một hộp 6
PASS ví dụ 2: phải dùng hết bốn hộp


### 12.2. Phản ví dụ cho các cách làm sai

Hai lỗi hay gặp: (a) **không sắp xếp** (cộng hộp theo thứ tự đầu vào); (b) **sắp xếp tăng dần** (lấy hộp nhỏ trước, dùng nhiều hộp hơn mức cần).

In [21]:
def wrong_no_sort(apple, capacity):
    total, running = sum(apple), 0
    for count, c in enumerate(capacity, start=1):          # SAI: không sắp xếp
        running += c
        if running >= total:
            return count


def wrong_ascending(apple, capacity):
    total, running = sum(apple), 0
    for count, c in enumerate(sorted(capacity), start=1):  # SAI: tăng dần, lấy hộp nhỏ trước
        running += c
        if running >= total:
            return count


bad = ([1, 3, 2], [1, 1, 5, 4, 3])       # tổng táo 6; đúng: 5 + 4 = 9 -> 2 hộp
print(wrong_no_sort(*bad), "!=", minimum_boxes_greedy(*bad))
print(wrong_ascending(*bad), "!=", minimum_boxes_greedy(*bad))
assert wrong_no_sort(*bad) != minimum_boxes_greedy(*bad)
assert wrong_ascending(*bad) != minimum_boxes_greedy(*bad)

3 != 2
4 != 2


### 12.3. Đối chiếu ngẫu nhiên (có seed)

Sinh input đúng ràng buộc (`n, m <= 50`, giá trị `1..50`, và tổng hộp `>=` tổng táo). Với `m` nhỏ so cả brute force; với `m` lớn so các cách nhanh với nhau.

In [22]:
def random_case(rng, n_max=50, m_max=50, hi=50):
    n = rng.randint(1, n_max)
    m = rng.randint(1, m_max)
    while True:
        apple = [rng.randint(1, hi) for _ in range(n)]
        capacity = [rng.randint(1, hi) for _ in range(m)]
        if sum(capacity) >= sum(apple):               # đề đảm bảo đặt được
            return apple, capacity
        n = max(1, n // 2)                            # giảm số gói để dễ thỏa


rng = random.Random(3074)
for trial in range(1500):
    apple, capacity = random_case(rng, rng.choice([3, 5, 10, 50]), rng.choice([2, 5, 8, 14]), rng.choice([3, 10, 50]))
    ref = minimum_boxes_bruteforce(list(apple), list(capacity))
    for key, fn in impls.items():
        assert fn(list(apple), list(capacity)) == ref, (key, apple, capacity)
print("PASS 1500 input nhỏ khớp brute force")

for trial in range(1500):
    apple, capacity = random_case(rng, 50, 50, rng.choice([5, 50]))
    assert 1 <= len(apple) <= 50 and 1 <= len(capacity) <= 50
    assert all(1 <= x <= 50 for x in apple + capacity)
    ref = minimum_boxes_greedy(list(apple), list(capacity))
    for key, fn in impls.items():
        assert fn(list(apple), list(capacity)) == ref, (key, apple, capacity)
print("PASS 1500 input lớn: các cách nhanh khớp nhau")

PASS 1500 input nhỏ khớp brute force


PASS 1500 input lớn: các cách nhanh khớp nhau


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên sắp xếp

In [23]:
def sai_khong_sap_xep(apple, capacity):
    total, running = sum(apple), 0
    for count, c in enumerate(capacity, start=1):
        running += c
        if running >= total:
            return count


apple, capacity = [1, 3, 2], [1, 1, 5, 4, 3]
print("sai:", sai_khong_sap_xep(apple, capacity), "| đúng:", Solution().minimumBoxes(apple, capacity))

sai: 3 | đúng: 2


**Vì sao sai:** cộng hộp theo thứ tự đầu vào là chọn hộp tùy ý, không đảm bảo ít hộp nhất. Với `capacity = [1, 1, 5, 4, 3]` và tổng táo `6`, code sai cộng `1 + 1 + 5 = 7 >= 6` nên ra `3`, còn đúng là `2` (hộp `5` và `4`). (Với input `[4, 3, 1, 5, 2]` của ví dụ 1 code sai tình cờ ra đúng, nên test kiểu đó không đủ để phát hiện lỗi.) **Cách sửa:** sắp xếp giảm dần.

### Lỗi 2: Sắp xếp tăng dần

In [24]:
def sai_tang_dan(apple, capacity):
    total, running = sum(apple), 0
    for count, c in enumerate(sorted(capacity), start=1):
        running += c
        if running >= total:
            return count


apple, capacity = [1, 3, 2], [4, 3, 1, 5, 2]
print("sai:", sai_tang_dan(apple, capacity), "| đúng:", Solution().minimumBoxes(apple, capacity))

sai: 3 | đúng: 2


**Vì sao sai:** lấy hộp **nhỏ trước** làm tổng sức chứa tăng chậm nên cần nhiều hộp hơn (`1 + 2 + 3 = 6` mới đủ, ra `3`). Muốn ít hộp thì phải lấy hộp **lớn trước**. **Cách sửa:** `reverse=True`.

### Lỗi 3: Dùng `>` thay vì `>=` khi kiểm tra đủ chỗ

In [25]:
def sai_lon_hon_hang(apple, capacity):
    total, running = sum(apple), 0
    for count, c in enumerate(sorted(capacity, reverse=True), start=1):
        running += c
        if running > total:             # SAI: bằng đúng total vẫn đủ
            return count


apple, capacity = [3, 3], [6, 1]
print("sai:", sai_lon_hon_hang(apple, capacity), "| đúng:", Solution().minimumBoxes(apple, capacity))

sai: 2 | đúng: 1


**Vì sao sai:** sức chứa **bằng** tổng số táo là vừa khít, hợp lệ. Dùng `>` ép thêm một hộp. **Cách sửa:** `running >= total`.

### Lỗi 4: Trả về chỉ số thay vì số lượng (lệch một)

In [26]:
def sai_tra_chi_so(apple, capacity):
    total, running = sum(apple), 0
    for i, c in enumerate(sorted(capacity, reverse=True)):      # i bắt đầu từ 0
        running += c
        if running >= total:
            return i                                            # SAI: số hộp là i + 1


apple, capacity = [1, 3, 2], [4, 3, 1, 5, 2]
print("sai:", sai_tra_chi_so(apple, capacity), "| đúng:", Solution().minimumBoxes(apple, capacity))

sai: 1 | đúng: 2


**Vì sao sai:** `enumerate` mặc định đếm từ `0`, nên chỉ số `i` ít hơn số hộp đã dùng `1`. **Cách sửa:** `enumerate(..., start=1)` hoặc trả `i + 1`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy "**chia tự do**" ⇒ chỉ cần **tổng**: gộp mọi thứ thành một đống rồi suy nghĩ.
- Thấy "ít … nhất để đạt tổng ≥ T" với các món độc lập ⇒ **tham lam: lấy món lớn nhất trước** + cộng dồn (cùng dạng với *Maximum Units on a Truck*, *Minimum Number of Boxes*).
- Giá trị bị chặn nhỏ (`<= 50`) ⇒ cân nhắc **mảng đếm** thay cho sắp xếp.
- Tổng tích lũy là tăng dần (với số dương) ⇒ có thể **tìm nhị phân** (`bisect`).
- Kiểm tra bằng **brute force nhỏ** để chắc tham lam đúng (như §6).

## 15. Biến thể của bài

1. **Mỗi gói phải nằm trọn trong một hộp** (không được chia): trở thành bài đóng gói (bin packing), khó hơn nhiều.
2. **Trả về chính các hộp được chọn** (danh sách sức chứa) thay vì số lượng.
3. **Hộp có chi phí** khác nhau: tối thiểu hóa chi phí (cần quy hoạch động / tham lam khác).
4. **Tối đa số táo đặt được khi chỉ có `k` hộp**: lấy `k` hộp lớn nhất (`min(total, tổng k lớn nhất)`).
5. **Maximum Units on a Truck (1710)**, **Boats to Save People (881)**: cùng họ tham lam theo sức chứa.

In [27]:
def chosen_boxes(apple, capacity):
    total = sum(apple)
    chosen, running = [], 0
    for c in sorted(capacity, reverse=True):
        chosen.append(c)
        running += c
        if running >= total:
            return chosen


def max_apples_with_k_boxes(apple, capacity, k):
    return min(sum(apple), sum(sorted(capacity, reverse=True)[:k]))


print(chosen_boxes([1, 3, 2], [4, 3, 1, 5, 2]))              # [5, 4]
print(max_apples_with_k_boxes([1, 3, 2], [4, 3, 1, 5, 2], 1))   # 5
assert chosen_boxes([1, 3, 2], [4, 3, 1, 5, 2]) == [5, 4]
assert max_apples_with_k_boxes([1, 3, 2], [4, 3, 1, 5, 2], 1) == 5
assert max_apples_with_k_boxes([1, 3, 2], [4, 3, 1, 5, 2], 3) == 6

[5, 4]
5


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử mọi tập hộp | định nghĩa, `combinations` | `O(2^m · m)` | `O(m)` | chỉ làm chuẩn kiểm tra cho `m` nhỏ |
| 2. Sắp xếp giảm dần + cộng dồn | tham lam | `O(n + m log m)` | `O(m)` | **nên nộp** |
| 3. Max-heap | lấy lớn nhất lặp lại | `O(n + m log m)` | `O(m)` | khi chỉ cần vài hộp đầu |
| 4. Đếm theo giá trị | `count[v]`, duyệt từ lớn xuống nhỏ | `O(n + m + C)` | `O(C)` | giá trị bị chặn nhỏ |
| 5. Tổng tích lũy + `bisect` | `bisect_left(prefix, total)` | `O(n + m log m)` | `O(m)` | khi cần trả lời nhiều truy vấn `total` |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `sorted([3, 1, 2], reverse=True)`, `list(accumulate([4, 3, 2]))`, `sum([2, 2, 2])`, `Solution().minimumBoxes([4], [1, 4, 2])`, `Solution().minimumBoxes([2, 2], [1, 1, 1, 1])`.

**Bài 2 (tính tay).** Với `apple = [7, 6, 5]`, `capacity = [3, 9, 4, 6, 2]`: tính tổng táo, sắp xếp hộp giảm dần, lập bảng tổng dồn, tìm số hộp tối thiểu.

**Bài 3 (điền TODO).** Hoàn thành `minimum_boxes_todo(apple, capacity)` theo Cách 2.

In [28]:
def minimum_boxes_todo(apple, capacity):
    # TODO 1: tính total = tổng số táo
    # TODO 2: sắp xếp capacity giảm dần (không sửa input)
    running = 0
    # TODO 3: duyệt từng hộp (đếm từ 1), cộng vào running
    #   TODO 4: nếu running >= total thì trả về số hộp đã dùng
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `minimum_boxes_counting2(apple, capacity)` theo Cách 4 (mảng đếm `count[0..50]`, duyệt giá trị từ `50` xuống `1`).

**Bài 5 (biến thể / mini challenge).** (a) Viết `chosen_cost(apple, capacity, cost)` trả **tổng chi phí `cost[i]`** của các hộp mà thuật toán tham lam (lớn trước) chọn. (b) Viết `max_apples(apple, capacity, k)` trả **số táo tối đa** đặt được khi chỉ được dùng `k` hộp.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `[3, 2, 1]`, `[4, 7, 9]`, `6`, `1`, `4`. Bài 3: `total = sum(apple)`, `for count, c in enumerate(sorted(capacity, reverse=True), start=1)`.
- **Mức 2 (vừa):** Bài 2: tổng táo `18`; hộp giảm dần `[9, 6, 4, 3, 2]`; tổng dồn `[9, 15, 19, 22, 24]`; `19 >= 18` đầu tiên ở vị trí thứ `3` ⇒ đáp án `3`. Bài 4: `count[c] += 1`; vòng `for v in range(50, 0, -1)` rồi `for _ in range(count[v])`.
- **Mức 3 (sâu):** Bài 5(a): lấy chỉ số các hộp theo thứ tự sức chứa giảm dần (`sorted(range(m), key=lambda i: -capacity[i])`), cộng sức chứa và chi phí song song cho tới khi đủ. 5(b): `min(sum(apple), tổng k hộp lớn nhất)`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [29]:
# Bài 1
assert sorted([3, 1, 2], reverse=True) == [3, 2, 1]
assert list(accumulate([4, 3, 2])) == [4, 7, 9]
assert sum([2, 2, 2]) == 6
assert Solution().minimumBoxes([4], [1, 4, 2]) == 1
assert Solution().minimumBoxes([2, 2], [1, 1, 1, 1]) == 4

# Bài 2
apple2, cap2 = [7, 6, 5], [3, 9, 4, 6, 2]
assert sum(apple2) == 18
assert sorted(cap2, reverse=True) == [9, 6, 4, 3, 2]
assert list(accumulate(sorted(cap2, reverse=True))) == [9, 15, 19, 22, 24]
assert Solution().minimumBoxes(apple2, cap2) == 3


# Bài 3
def minimum_boxes_todo_ans(apple, capacity):
    total = sum(apple)
    running = 0
    for count, c in enumerate(sorted(capacity, reverse=True), start=1):
        running += c
        if running >= total:
            return count


for apple, capacity, expected in examples:
    assert minimum_boxes_todo_ans(apple, capacity) == expected


# Bài 4
def minimum_boxes_counting2(apple, capacity):
    total = sum(apple)
    count = [0] * 51
    for c in capacity:
        count[c] += 1
    running = boxes = 0
    for v in range(50, 0, -1):
        for _ in range(count[v]):
            running += v
            boxes += 1
            if running >= total:
                return boxes


for apple, capacity, expected in examples:
    assert minimum_boxes_counting2(apple, capacity) == expected


# Bài 5(a): tổng chi phí của các hộp được tham lam chọn
def chosen_cost(apple, capacity, cost):
    total = sum(apple)
    order = sorted(range(len(capacity)), key=lambda i: -capacity[i])
    running = spent = 0
    for i in order:
        running += capacity[i]
        spent += cost[i]
        if running >= total:
            return spent


assert chosen_cost([1, 3, 2], [4, 3, 1, 5, 2], [4, 3, 1, 5, 2]) == 9              # hộp 5 và 4
assert chosen_cost([1, 3, 2], [4, 3, 1, 5, 2], [10, 10, 10, 7, 10]) == 17        # hộp 5 (chi phí 7) và hộp 4 (chi phí 10)


# Bài 5(b)
def max_apples(apple, capacity, k):
    return min(sum(apple), sum(sorted(capacity, reverse=True)[:k]))


assert max_apples([1, 3, 2], [4, 3, 1, 5, 2], 1) == 5
assert max_apples([1, 3, 2], [4, 3, 1, 5, 2], 2) == 6
assert max_apples([10], [3, 2], 2) == 5
print("PASS toàn bộ đáp án")

PASS toàn bộ đáp án


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (chọn ít món nhất có tổng `>= T`: sắp xếp giảm dần rồi cộng dồn)

```python
from typing import List

class Solution:
    def minimumBoxes(self, apple: List[int], capacity: List[int]) -> int:
        total = sum(apple)
        running = 0
        for count, c in enumerate(sorted(capacity, reverse=True), start=1):
            running += c
            if running >= total:
                return count
        return -1
```

**Checklist**

- [ ] Đã tính `total = sum(apple)` (táo chia tự do nên chỉ cần tổng)?
- [ ] Sắp xếp hộp **giảm dần** (lớn trước)?
- [ ] Điều kiện dừng là `running >= total` (không phải `>`)?
- [ ] Trả về **số hộp** (`start=1` hoặc `i + 1`), không phải chỉ số?
- [ ] Đã thử tổng táo đúng bằng sức chứa một hộp, và trường hợp phải dùng hết hộp?
- [ ] Skeleton đúng `class Solution` / `def minimumBoxes(self, apple: List[int], capacity: List[int]) -> int`?